# **Regression: Lasso Regression (L1 Regularization)**

## **Justification of Preprocessing Strategy**

### **The Absolute Necessity of Scaling for L1 Penalty**
**Lasso Regression** introduces an L1 regularization penalty to the Ordinary Least Squares (OLS) loss function, adding the sum of the absolute values of the coefficients to the optimization target.

Because the penalty directly restricts the absolute magnitude of the $\beta$ weights, features with larger underlying numerical ranges will naturally have smaller coefficients to compensate, making them the target of unfair penalization. To prevent the algorithm from erroneously suppressing clinically significant variables, the feature space must be uniform. We will evaluate both **Standardization (StandardScaler)** and **Normalization (MinMaxScaler)** across all optimization levels to discover which framework yields the most accurate predictions for the continuous `diabetes_risk_score`.

### **Automated Feature Selection and Sparsity**
The unique mathematical trait of L1 regularization is its ability to force less important feature coefficients to become **exactly zero**. In a dataset containing 100,000 samples and numerous dummy-encoded variables, Lasso acts as an embedded feature selection tool, filtering out noise and tackling multicollinearity. To avoid data leakage, we drop categorical targets (`diagnosed_diabetes`, `diabetes_stage`), and we **omit stratification** during the split since we are modeling a continuous numerical distribution.


## **Experiment Design**

We have designed a complete tournament consisting of **6 distinct runs** to evaluate both scalers across 3 optimization levels, using **MAE, RMSE, and $R^2$** as performance indicators:

* **Run 1 & 2: Lasso Baseline** — Testing the Lasso model with strict Scikit-Learn default parameters under **Standardization** vs. **Normalization**.
* **Run 3 & 4: GridSearchCV Tuning** — Performing an exhaustive search over a fixed grid of the `alpha` parameter under **Standardization** vs. **Normalization**.
* **Run 5 & 6: Optuna Optimization** — Utilizing Bayesian optimization to fine-tune both `alpha` and `max_iter` continuously under **Standardization** vs. **Normalization**.

In all optimization runs (GridSearchCV and Optuna), trials are evaluated using **3-Fold Cross-Validation** to guarantee model generalizability.


In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Lasso")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Drop classification targets to avoid data leakage
X = df_final.drop(["diabetes_risk_score", "diagnosed_diabetes", "diabetes_stage"], axis=1, errors='ignore')
y = df_final['diabetes_risk_score']

# Split data (80/20) - Continuous target means NO stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_regression_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to evaluate Overfitting/Underfitting"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr, y_tr_pred))
    mlflow.log_metric("rmse_train", np.sqrt(mean_squared_error(y_tr, y_tr_pred)))
    mlflow.log_metric("r2_train", r2_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("mae_test", mean_absolute_error(y_te, y_te_pred))
    mlflow.log_metric("rmse_test", np.sqrt(mean_squared_error(y_te, y_te_pred)))
    mlflow.log_metric("r2_test", r2_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# Scaling strategies to compare across the entire tournament
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

# ---------------------------------------------------------
# RUN 1 & 2: LASSO BASELINE (Strict Defaults)
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Lasso_Baseline_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        model = Lasso(random_state=42)
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        mlflow.log_param("optimization", "none_default")
        mlflow.log_param("scaler", s_name)
        mlflow.log_params(model.get_params())
        
        log_regression_metrics(model, X_train_scaled, y_train, X_test_scaled, y_test, duration)

# ---------------------------------------------------------
# RUN 3 & 4: GRIDSEARCHCV TUNING
# ---------------------------------------------------------
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]}

for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Lasso_GridSearch_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        grid = GridSearchCV(
            Lasso(random_state=42, max_iter=2000), 
            param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        best_model = grid.best_estimator_
        zero_coefs = np.sum(best_model.coef_ == 0)
        
        mlflow.log_param("optimization", "GridSearchCV")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("eliminated_features", f"{zero_coefs}/{len(best_model.coef_)}")
        mlflow.log_params(grid.best_params_)
        
        log_regression_metrics(best_model, X_train_scaled, y_train, X_test_scaled, y_test, duration)

# ---------------------------------------------------------
# RUN 5 & 6: OPTUNA BAYESIAN OPTIMIZATION
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
    
    def objective(trial):
        alpha_val = trial.suggest_float("alpha", 1e-5, 5.0, log=True)
        max_iter_val = trial.suggest_int("max_iter", 1000, 4000)
        model = Lasso(alpha=alpha_val, max_iter=max_iter_val, random_state=42)
        scores = -cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1).mean()
        return scores.mean()

    with mlflow.start_run(run_name=f"Lasso_Optuna_{s_name}"):
        study = optuna.create_study(direction="minimize")
        start_time = time.time()
        study.optimize(objective, n_trials=15)
        duration = time.time() - start_time
        
        best_lasso_opt = Lasso(**study.best_params, random_state=42)
        best_lasso_opt.fit(X_train_scaled, y_train)
        zero_coefs_opt = np.sum(best_lasso_opt.coef_ == 0)
        
        mlflow.log_param("optimization", "optuna")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("eliminated_features", f"{zero_coefs_opt}/{len(best_lasso_opt.coef_)}")
        mlflow.log_params(study.best_params)
        
        log_regression_metrics(best_lasso_opt, X_train_scaled, y_train, X_test_scaled, y_test, duration)

[I 2026-05-20 13:29:33,078] A new study created in memory with name: no-name-25eafce3-53ca-44f4-9651-bde6de63d101
[I 2026-05-20 13:29:34,167] Trial 0 finished with value: 0.39708817103800637 and parameters: {'alpha': 0.00031441181972483923, 'max_iter': 3434}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:34,987] Trial 1 finished with value: 0.4049854803690766 and parameters: {'alpha': 0.010948581690550014, 'max_iter': 2849}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:35,810] Trial 2 finished with value: 0.4009191363716474 and parameters: {'alpha': 0.006350009784234054, 'max_iter': 2725}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:36,656] Trial 3 finished with value: 0.39718569421438854 and parameters: {'alpha': 0.0005871830271953236, 'max_iter': 2146}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:37,919] Trial 4 finished with value: 0.3970687028051461 and parameters: {'alpha': 0.000244850

## **Winner Run Selection (Priority Elimination Framework)**

### **Selection Criteria (in priority order)**
1. **Priority 1 (60% weight): Lowest MAE (Test)** — Clinical proximity; minimizes average day-to-day prediction error on unseen data.
2. **Priority 2 (30% weight): RMSE proportional to MAE** — Rejects runs where RMSE spikes relative to MAE, indicating catastrophic errors.
3. **Priority 3 (10% weight): Acceptable R² (Test)** — Confirms statistical fit quality on unseen data.
4. **Tiebreaker: Lowest Fit Time** — Applied only if a technical tie exists in MAE, RMSE, and R² metrics.

### **All Runs: Summary Table with Train and Test Metrics**

| Run | Eliminated Features | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time (s) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Lasso_Optuna_Standardization | 4/53 | 0.3966757837 | 0.4038933897 | 0.6936680357 | 0.7112917067 | 0.9941322827 | 0.9938694744 | 46.4559938908 |
| **Lasso_GridSearch_Standardization** | **6/53** | **0.3967013490** | **0.4039317592** | **0.6936728172** | **0.7112927860** | **0.9941322018** | **0.9938694558** | **26.3300929070** |
| Lasso_Optuna_Normalization | 5/53 | 0.3967142664 | 0.4039546474 | 0.6936703127 | 0.7113068971 | 0.9941322442 | 0.9938692125 | 21.1862425804 |
| Lasso_GridSearch_Normalization | 12/53 | 0.3971401546 | 0.4043579826 | 0.6937018316 | 0.7112789223 | 0.9941317110 | 0.9938696947 | 4.7376437187 |
| Lasso_Baseline_Standardization | 0/53 | 2.0091024164 | 2.0209383078 | 2.5151186299 | 2.5227186289 | 0.9228594445 | 0.9228848245 | 0.1960139275 |
| Lasso_Baseline_Normalization | 0/53 | 5.0415100652 | 5.0730118293 | 6.2955069526 | 6.3321558577 | 0.5166883880 | 0.5141464116 | 0.1650586128 |

### **Step-by-Step Elimination Process**

**Step 1: Filter by Lowest Test MAE (Priority 1 — 60%)**
- Threshold: Test MAE ≤ 0.4038933897
- Candidates passing: Lasso_Optuna_Standardization (0.4038933897)
- Status: Optuna_Standardization is the best test-MAE run; all other runs are slightly worse.

**Step 2: Verify RMSE Proportional to MAE (Priority 2 — 30%)**
- Winner candidate: RMSE (Test) = 0.7112917067
- MAE-to-RMSE ratio: 0.4038933897 / 0.7112917067 ≈ 0.568
- Status: **No catastrophic divergence detected**. The winner candidate passes.

**Step 3: Confirm Acceptable Test R² (Priority 3 — 10%)**
- Winner candidate: R² (Test) = 0.9938694744
- Status: Excellent fit confirmed.

**Step 4: Apply Tiebreaker — Lowest Fit Time**
- Not required: there is no tie after comparing test MAE, RMSE, and R².

### **Final Decision**
**Winner: Lasso_Optuna_Standardization**

**Justification:** Lasso_Optuna_Standardization delivers the lowest test MAE among all runs, while also preserving an excellent RMSE and R² profile. Although GridSearch_Standardization is faster, the tiebreaker is not used here because the runs are not tied on the primary performance criteria. This makes Optuna_Standardization the best overall choice for predictive quality.

### **Winner Hyperparameters**

| Parameter | Value |
|---|---|
| **alpha** | 4.427123816536028e-05 |
| **max_iter** | 1131 |
| **random_state** | 42 |
| **eliminated_features** | 4/53 |

## **Overfitting/Underfitting Diagnosis (Evidence-Based Analysis)**

### **Train vs Test Gap Analysis**

| Metric | Train | Test | Gap | Interpretation |
|---|---:|---:|---:|---|
| **MAE** | 0.3966757837 | 0.4038933897 | +0.0072176060 (+1.82%) | Minimal gap; test error is only slightly higher than train |
| **RMSE** | 0.6936680357 | 0.7112917067 | +0.0176236710 (+2.54%) | Minimal gap; test error increases slightly |
| **R² Score** | 0.9941322827 | 0.9938694744 | −0.0002628083 (−0.03%) | Negligible degradation on test data |

### **Diagnosis: No Evidence of Overfitting or Underfitting in the Winner**

**Rationale:**
- **Train-Test Gaps are Minimal**: The differences between train and test metrics are very small, which indicates strong generalization.
- **No Significant Performance Degradation**: There is no meaningful drop from train to test, so the winner does not show a classic overfitting pattern.
- **Excellent Fit Quality**: Both train and test R² are above 0.99, which is a strong sign of a well-fitted model.
- **No Underfitting Signal in the Winner**: Low errors and very high R² on both partitions show that the selected model is not too simplistic.

### **Additional Note on the Baselines**
- The baseline runs, especially **Lasso_Baseline_Normalization**, show much worse errors and lower R², which is a clear sign of **underfitting due to excessive regularization**.
- The tuned runs correct that behavior and converge to very strong test performance.

### **Conclusion**
The selected Lasso model, **Lasso_Optuna_Standardization**, shows **excellent generalization behavior** with no signs of overfitting or underfitting. The baseline configurations underfit, but the optimized model is well balanced and predictive on both training and test data.